# Module 6.5 — Packaging & Distribution

### Python Mastery · Track 6: Testing, Tooling & DevOps

To share code so others can install and use it, you package it. This module covers isolating dependencies with virtual environments, declaring a project with `pyproject.toml`, building a distributable wheel, and the path to publishing on the Python Package Index.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- The examples create real files and a real virtual environment in the working directory, and build an actual package, so you see genuine output. Publishing to PyPI is described rather than performed, since it requires an account and would upload to the public index.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Create and use a virtual environment for dependency isolation.
2. Record and install dependencies with `pip` and `requirements.txt`.
3. Describe a project with a modern `pyproject.toml`.
4. Build a wheel and source distribution.
5. Describe how a package is published and the role of lock files.

**Prerequisites:** Tracks 1 to 5.

---

## Part 1 · Virtual Environments

Different projects need different, sometimes conflicting, package versions. A **virtual environment** is an isolated Python installation for one project, so its dependencies do not clash with other projects or the system Python. You create one with the standard `venv` module and "activate" it to use it.

The cell below creates a real virtual environment in a temporary folder and confirms it has its own interpreter and `pip`.

In [ ]:
import subprocess, sys, os, shutil

workdir = "demo_env"
shutil.rmtree(workdir, ignore_errors=True)

# Create a virtual environment named 'demo_env'.
subprocess.run([sys.executable, "-m", "venv", workdir], check=True)

# Inside it is a self-contained interpreter and pip (path differs by platform).
bin_dir = "Scripts" if os.name == "nt" else "bin"
venv_python = os.path.join(workdir, bin_dir, "python")
print("created virtual environment at:", workdir)
print("its interpreter:", venv_python)
print("contents:", sorted(os.listdir(os.path.join(workdir, bin_dir)))[:6], "...")

In a terminal you would **activate** the environment (`source demo_env/bin/activate` on Linux or macOS, `demo_env\\Scripts\\activate` on Windows). Activation puts the environment's `python` and `pip` first on your path, so installs and runs use the isolated environment rather than the system one. Deactivate with `deactivate`.

## Part 2 · Dependencies with `pip` and `requirements.txt`

`pip` installs packages from the Python Package Index. To make an environment reproducible, you record the exact packages and versions in a `requirements.txt` file, which others (and your future self, and your servers) can install with one command. Pinning versions ensures everyone gets the same dependencies.

In [ ]:
# A requirements file lists each dependency, ideally with a pinned version.
requirements = """requests==2.32.3
rich>=13.0
python-dateutil
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt:")
print(open("requirements.txt").read())
print("Install them all with:  pip install -r requirements.txt")
print("(== pins an exact version; >= sets a minimum; no marker takes the latest compatible)")

A related file, `constraints.txt`, pins versions without requiring installation, useful for controlling the versions of indirect dependencies. To capture exactly what is currently installed, `pip freeze` prints every package and its version, which you can redirect into a requirements file.

## Part 3 · Describing a Project with `pyproject.toml`

Modern Python projects are described by a single `pyproject.toml` file: it names the project, its version, its dependencies, and which build tool to use. This has replaced the older `setup.py` for most purposes. The cell below writes a complete, minimal package: the configuration plus a small source module.

In [ ]:
import os

# Create the package source layout: a 'src' directory containing the package.
os.makedirs("mytools/src/mytools", exist_ok=True)

# The package's code. (A real package would use a module docstring; we use a
# comment here so the generated content stays simple.)
init_code = """# A tiny example package.

__version__ = "0.1.0"

def shout(text):
    # Return the text in uppercase with an exclamation mark.
    return text.upper() + "!"
"""
with open("mytools/src/mytools/__init__.py", "w") as f:
    f.write(init_code)

# The project description and build configuration.
pyproject = """[build-system]
requires = ["setuptools>=61.0"]
build-backend = "setuptools.build_meta"

[project]
name = "mytools"
version = "0.1.0"
description = "A tiny example package"
readme = "README.md"
requires-python = ">=3.9"
dependencies = []

[project.optional-dependencies]
dev = ["pytest"]
"""
with open("mytools/pyproject.toml", "w") as f:
    f.write(pyproject)

with open("mytools/README.md", "w") as f:
    f.write("# mytools\n\nA tiny example package for the course.\n")

print("project files created:")
for root, _, files in os.walk("mytools"):
    for name in sorted(files):
        print(" ", os.path.join(root, name))

In [ ]:
# Show the project description.
print(open("mytools/pyproject.toml").read())

## Part 4 · Building a Distribution

To share the package, you **build** it into distributable files: a **wheel** (`.whl`, a ready-to-install binary format) and a **source distribution** (`.tar.gz`). The standard tool is `build`, run as `python -m build`. The cell below builds the package we just created and lists the resulting files.

In [ ]:
import subprocess, sys, os

# Install the build tool if needed, then build the package.
subprocess.run([sys.executable, "-m", "pip", "install", "build", "--quiet",
                "--break-system-packages"], check=False)

result = subprocess.run(
    [sys.executable, "-m", "build", "mytools"],
    capture_output=True, text=True,
)
print("build finished with return code:", result.returncode)

dist = os.path.join("mytools", "dist")
if os.path.isdir(dist):
    print("built artifacts:")
    for name in sorted(os.listdir(dist)):
        print("  ", name)        # a .whl and a .tar.gz

The `.whl` file can be installed directly with `pip install path/to/file.whl`, and is what `pip install yourpackage` downloads from the index. The `.tar.gz` source distribution lets others build the package themselves. Both are produced from the single `pyproject.toml`.

## Part 5 · Publishing, and Lock Files

To make a package installable by anyone, you upload the built files to the **Python Package Index (PyPI)** with a tool such as `twine`: `twine upload dist/*`. The strong recommendation is to publish first to **TestPyPI**, a separate sandbox index, to verify everything before the real upload, since a version on PyPI cannot be replaced.

For applications (as opposed to libraries), a **lock file** records the exact version of every dependency, including indirect ones, so installs are perfectly reproducible. Tools such as `pip-tools`, Poetry, and the newer `uv` generate and manage lock files. The cell below summarises the workflow as a checklist rather than performing an upload.

In [ ]:
steps = [
    "1. Describe the project in pyproject.toml (name, version, dependencies).",
    "2. Build with:            python -m build",
    "3. Check the artifacts:   twine check dist/*",
    "4. Upload to TestPyPI:    twine upload --repository testpypi dist/*",
    "5. Verify installation:   pip install --index-url https://test.pypi.org/... yourpackage",
    "6. Upload to PyPI:        twine upload dist/*",
]
print("Publishing checklist (perform in a terminal, not here):")
for step in steps:
    print("  " + step)

print()
print("Lock files (pip-tools, Poetry, uv) pin EVERY dependency, including indirect")
print("ones, so an application installs identically everywhere. A version published")
print("to PyPI is permanent and cannot be overwritten, so test on TestPyPI first.")

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Writing a requirements file (Easy)

In [ ]:
content = "requests==2.32.3\nrich>=13.0\n"
with open("ex_requirements.txt", "w") as f:
    f.write(content)
print(open("ex_requirements.txt").read())
print("Install with: pip install -r ex_requirements.txt")

### Example 2 — Creating a virtual environment (Easy)

In [ ]:
import subprocess, sys, os, shutil
shutil.rmtree("ex_env", ignore_errors=True)
subprocess.run([sys.executable, "-m", "venv", "ex_env"], check=True)
bin_dir = "Scripts" if os.name == "nt" else "bin"
print("environment created; interpreter at:", os.path.join("ex_env", bin_dir, "python"))

### Example 3 — A minimal pyproject.toml (Medium)

In [ ]:
toml = """[build-system]
requires = ["setuptools>=61.0"]
build-backend = "setuptools.build_meta"

[project]
name = "greeter"
version = "0.0.1"
description = "Says hello"
requires-python = ">=3.9"
dependencies = []
"""
with open("ex_pyproject.toml", "w") as f:
    f.write(toml)
print(open("ex_pyproject.toml").read())

### Example 4 — Capturing installed packages (Medium)

In [ ]:
import subprocess, sys
# pip freeze prints currently installed packages with exact versions.
result = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                        capture_output=True, text=True)
lines = result.stdout.strip().splitlines()
print(f"{len(lines)} packages installed; first five:")
for line in lines[:5]:
    print("  ", line)
print("Redirect this into a file with: pip freeze > requirements.txt")

### Example 5 — Building a package end to end (Difficult)

In [ ]:
import os, subprocess, sys

# Create a small package.
os.makedirs("greeterpkg/src/greeter", exist_ok=True)
with open("greeterpkg/src/greeter/__init__.py", "w") as f:
    f.write('def hello(name):\n    return f"Hello, {name}!"\n')
pyproject = """[build-system]
requires = ["setuptools>=61.0"]
build-backend = "setuptools.build_meta"

[project]
name = "greeter"
version = "0.1.0"
requires-python = ">=3.9"
"""
with open("greeterpkg/pyproject.toml", "w") as f:
    f.write(pyproject)

# Build it.
result = subprocess.run([sys.executable, "-m", "build", "greeterpkg"],
                        capture_output=True, text=True)
print("build return code:", result.returncode)
dist = "greeterpkg/dist"
if os.path.isdir(dist):
    print("artifacts:", sorted(os.listdir(dist)))

### Example 6 — Inspecting a built wheel (Difficult)

In [ ]:
import os, zipfile

# A wheel is a ZIP archive; we can list its contents to see what was packaged.
dist = "greeterpkg/dist"
wheels = [f for f in os.listdir(dist) if f.endswith(".whl")] if os.path.isdir(dist) else []
if wheels:
    wheel_path = os.path.join(dist, wheels[0])
    print("inspecting:", wheels[0])
    with zipfile.ZipFile(wheel_path) as z:
        for name in z.namelist():
            print("  ", name)
else:
    print("no wheel found; run Example 5 first")

---

## Exercises

**Exercise 1 (Easy).** Write a `requirements.txt` listing two dependencies, one with a pinned version and one with a minimum version, then print the file.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Create a virtual environment named `practice_env` using `venv`, and print the path to its interpreter.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Write a minimal `pyproject.toml` for a project named `calc` at version `0.1.0` requiring Python 3.9 or later, then print it.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Use `pip freeze` (via `subprocess`) to count how many packages are installed, and print the count.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Create a tiny package (a `pyproject.toml` plus a source module with one function) and build it with `python -m build`, then list the files produced in its `dist` directory.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
content = "flask==3.0.0\nnumpy>=1.26\n"
with open("sol1_requirements.txt", "w") as f:
    f.write(content)
print(open("sol1_requirements.txt").read())

**Solution 2**

In [ ]:
import subprocess, sys, os, shutil
shutil.rmtree("practice_env", ignore_errors=True)
subprocess.run([sys.executable, "-m", "venv", "practice_env"], check=True)
bin_dir = "Scripts" if os.name == "nt" else "bin"
print(os.path.join("practice_env", bin_dir, "python"))

**Solution 3**

In [ ]:
toml = """[build-system]
requires = ["setuptools>=61.0"]
build-backend = "setuptools.build_meta"

[project]
name = "calc"
version = "0.1.0"
requires-python = ">=3.9"
dependencies = []
"""
with open("sol3_pyproject.toml", "w") as f:
    f.write(toml)
print(open("sol3_pyproject.toml").read())

**Solution 4**

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                        capture_output=True, text=True)
print("installed packages:", len(result.stdout.strip().splitlines()))

**Solution 5**

In [ ]:
import os, subprocess, sys
os.makedirs("sol5pkg/src/tinytool", exist_ok=True)
with open("sol5pkg/src/tinytool/__init__.py", "w") as f:
    f.write('def double(n):\n    return n * 2\n')
pyproject = """[build-system]
requires = ["setuptools>=61.0"]
build-backend = "setuptools.build_meta"

[project]
name = "tinytool"
version = "0.1.0"
requires-python = ">=3.9"
"""
with open("sol5pkg/pyproject.toml", "w") as f:
    f.write(pyproject)
result = subprocess.run([sys.executable, "-m", "build", "sol5pkg"],
                        capture_output=True, text=True)
print("build return code:", result.returncode)
if os.path.isdir("sol5pkg/dist"):
    print("artifacts:", sorted(os.listdir("sol5pkg/dist")))

---

## Summary and Key Points

- A virtual environment (`python -m venv name`) isolates a project's dependencies; activation puts its `python` and `pip` first on the path.
- `pip` installs packages; `requirements.txt` records them (pin exact versions with `==`), and `pip freeze` captures the current set. `constraints.txt` pins versions without installing.
- `pyproject.toml` describes a modern project, its metadata, dependencies, and build backend, replacing `setup.py` for most uses.
- `python -m build` produces a wheel (`.whl`, ready to install) and a source distribution (`.tar.gz`) from that single configuration.
- Publishing uploads the artifacts to PyPI with `twine` (test on TestPyPI first, as releases are permanent); lock files via pip-tools, Poetry, or uv pin every dependency for reproducible application installs.

### Next module: 6.6 — CI/CD & DevOps

The next module covers automating the build-test-check cycle: continuous integration with workflow files, multi-environment testing with tox and nox, containerising with Docker, and managing secrets.